In [40]:
#AI 私厨
from dotenv import load_dotenv
load_dotenv()
#定义模型
from langchain.chat_models import init_chat_model

model = init_chat_model("deepseek-chat")

In [41]:
#定义工具
from langchain_tavily import TavilySearch

#web搜索工具
web_search = TavilySearch(
    max_results = 5,
    topic = "general",
)

In [42]:
#记忆管理
#sqlite

from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 连接sqlite
connection = sqlite3.connect("D:/Python-Project/notebook/resources/personal_chief.db", check_same_thread=False)
# 初始化checkpointer
checkpointer = SqliteSaver(connection)
# 自动建表
checkpointer.setup()

In [43]:
#创建agent
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作：
1.识别和评估食材：若用户提供照片，首先辨识所有可见食材。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”。
2.智能食谱检索：优先调用 web_search 工具，以“可用食材清单”为核心关键词，查找可行菜谱。
3.多维度评估与排序：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，制作简单且营养丰富的排名靠前。
4.结构化方案输出：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策。

请严格按照流程，优先调用 web_search 工具搜索食谱，搜索不到的情况下才能自己发挥。
"""

agent = create_agent(
    model= model,
    system_prompt= system_prompt,
    tools= [web_search],
    checkpointer= checkpointer,
)

In [45]:
#测试
#准备多模态信息
from langchain.messages import HumanMessage

# 准备多模态消息，图片是网络上的冰箱食物图片
multimodal_message = HumanMessage(
    content=[
        {"type": "image",
         "url": "https://tse1.mm.bing.net/th/id/OIP.XW0rg7UEohUNvAOyIvuUiwHaE7?r=0&rs=1&pid=ImgDetMain&o=7&rm=3"},
        {"type": "text", "text": "帮我看看这些食材能做些什么？"}
    ])

config = {"configurable": {"thread_id": "ingredients_clean_test_01"}}

response = agent.invoke({"messages": [multimodal_message]}, config)

In [46]:
# 友好打印
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://tse1.mm.bing.net/th/id/OIP.XW0rg7UEohUNvAOyIvuUiwHaE7?r=0&rs=1&pid=ImgDetMain&o=7&rm=3'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================

我先看一下这张食材图，然后根据里面可用的食材去网上搜一搜合适的菜谱，最后再帮你做一个兼顾做法简单和营养丰富的排序和推荐。
Tool Calls:
  tavily_search (call_00_KLB1LdcmW4hXiwjPm1Sl0046)
 Call ID: call_00_KLB1LdcmW4hXiwjPm1Sl0046
  Args:
    query: 紫甘蓝 胡萝卜 小番茄 黄瓜 生菜 菜谱 做法
    search_depth: basic
    include_images: True
  tavily_search (call_01_Fv93skE2B3A44EcC8hpr2972)
 Call ID: call_01_Fv93skE2B3A44EcC8hpr2972
  Args:
    query: 紫甘蓝 胡萝卜 番茄 沙拉 食谱 营养
    search_depth: basic
    include_images: True
  tavily_search (call_02_6qCq3xtZiZxj7g3IbIrl5268)
 Call ID: call_02_6qCq3xtZiZxj7g3IbIrl5268
  Args:
    query: 小番茄 黄瓜 生菜 胡萝卜 凉拌菜 简单做法
    search_depth: basic
    include_images: True
================================= Tool Me

In [48]:
response = agent.invoke({"messages": [HumanMessage("两个人吃，当主食吃")]}, config)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://tse1.mm.bing.net/th/id/OIP.XW0rg7UEohUNvAOyIvuUiwHaE7?r=0&rs=1&pid=ImgDetMain&o=7&rm=3'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================

我先看一下这张食材图，然后根据里面可用的食材去网上搜一搜合适的菜谱，最后再帮你做一个兼顾做法简单和营养丰富的排序和推荐。
Tool Calls:
  tavily_search (call_00_KLB1LdcmW4hXiwjPm1Sl0046)
 Call ID: call_00_KLB1LdcmW4hXiwjPm1Sl0046
  Args:
    query: 紫甘蓝 胡萝卜 小番茄 黄瓜 生菜 菜谱 做法
    search_depth: basic
    include_images: True
  tavily_search (call_01_Fv93skE2B3A44EcC8hpr2972)
 Call ID: call_01_Fv93skE2B3A44EcC8hpr2972
  Args:
    query: 紫甘蓝 胡萝卜 番茄 沙拉 食谱 营养
    search_depth: basic
    include_images: True
  tavily_search (call_02_6qCq3xtZiZxj7g3IbIrl5268)
 Call ID: call_02_6qCq3xtZiZxj7g3IbIrl5268
  Args:
    query: 小番茄 黄瓜 生菜 胡萝卜 凉拌菜 简单做法
    search_depth: basic
    include_images: True
================================= Tool Me

In [ ]:
response = agent.invoke({"messages": [HumanMessage("你还记得我们几个人吃吗")]}, config)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'image', 'url': 'https://tse1.mm.bing.net/th/id/OIP.XW0rg7UEohUNvAOyIvuUiwHaE7?r=0&rs=1&pid=ImgDetMain&o=7&rm=3'}, {'type': 'text', 'text': '帮我看看这些食材能做些什么？'}]
================================== Ai Message ==================================

我先看一下这张食材图，然后根据里面可用的食材去网上搜一搜合适的菜谱，最后再帮你做一个兼顾做法简单和营养丰富的排序和推荐。
Tool Calls:
  tavily_search (call_00_KLB1LdcmW4hXiwjPm1Sl0046)
 Call ID: call_00_KLB1LdcmW4hXiwjPm1Sl0046
  Args:
    query: 紫甘蓝 胡萝卜 小番茄 黄瓜 生菜 菜谱 做法
    search_depth: basic
    include_images: True
  tavily_search (call_01_Fv93skE2B3A44EcC8hpr2972)
 Call ID: call_01_Fv93skE2B3A44EcC8hpr2972
  Args:
    query: 紫甘蓝 胡萝卜 番茄 沙拉 食谱 营养
    search_depth: basic
    include_images: True
  tavily_search (call_02_6qCq3xtZiZxj7g3IbIrl5268)
 Call ID: call_02_6qCq3xtZiZxj7g3IbIrl5268
  Args:
    query: 小番茄 黄瓜 生菜 胡萝卜 凉拌菜 简单做法
    search_depth: basic
    include_images: True
================================= Tool Me

: 